<a href="https://colab.research.google.com/github/vishakha122/dissertation/blob/main/mimic_datasets(load_and_merge).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MIMIC-IV Hybrid Disposition Prediction - Project Pipeline
Overview <br>
Building a hybrid neural network to predict ED patient disposition (ADMITTED/DISCHARGED/NON-STANDARD) using MIMIC-IV-ED structured data + clinical notes via ClinicalBERT + MLP fusion architecture.<br>
# Notebook 1: Data Loading & Merging
Purpose: Load and merge MIMIC-IV-ED structured data and clinical notes
Inputs:<br>
MIMIC-IV-ED tables (ed/*.csv.gz)<br>
MIMIC-IV clinical notes (note/*.csv.gz)<br>
Outputs:<br>
structured_merged.csv: All ED data merged by stay_id<br>
notes_merged.csv: Clinical notes linked to stays<br>



In [ ]:
#Mount Google Drive and set up directories
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#install command line utility to retrive content and download files from web servers
!pip install wget

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=460476c355e9ead43c7fa4e5a84e5dde36ecbc540d0f57c3a831887bb4e82149
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget


In [ ]:
!wget -r -N -c -np --user vishakha-hwu04 --ask-password https://physionet.org/files/mimic-iv-note/2.2/
#command to download and use the necessary datasets from physionet.org

Password for user ‘vishakha-hwu04’: 
--2026-03-22 10:05:31--  https://physionet.org/files/mimic-iv-note/2.2/
Resolving physionet.org (physionet.org)... 18.18.42.54
Connecting to physionet.org (physionet.org)|18.18.42.54|:443... connected.
HTTP request sent, awaiting response... 401 Unauthorized
Authentication selected: Basic realm="PhysioNet", charset="UTF-8"
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘physionet.org/files/mimic-iv-note/2.2/index.html’

physionet.org/files     [ <=>                ]     574  --.-KB/s    in 0s      

Last-modified header missing -- time-stamps turned off.
2026-03-22 10:05:31 (88.4 MB/s) - ‘physionet.org/files/mimic-iv-note/2.2/index.html’ saved [574]

Loading robots.txt; please ignore errors.
--2026-03-22 10:05:31--  https://physionet.org/robots.txt
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 200 OK

    The file 

In [ ]:
!wget -r -N -c -np --user vishakha-hwu04 --ask-password https://physionet.org/files/mimic-iv-ed/2.2/

Password for user ‘vishakha-hwu04’: 
--2026-03-22 13:49:11--  https://physionet.org/files/mimic-iv-ed/2.2/
Resolving physionet.org (physionet.org)... 18.18.42.54
Connecting to physionet.org (physionet.org)|18.18.42.54|:443... connected.
HTTP request sent, awaiting response... 401 Unauthorized
Authentication selected: Basic realm="PhysioNet", charset="UTF-8"
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘physionet.org/files/mimic-iv-ed/2.2/index.html’

physionet.org/files     [ <=>                ]     683  --.-KB/s    in 0s      

Last-modified header missing -- time-stamps turned off.
2026-03-22 13:49:12 (108 MB/s) - ‘physionet.org/files/mimic-iv-ed/2.2/index.html’ saved [683]

Loading robots.txt; please ignore errors.
--2026-03-22 13:49:12--  https://physionet.org/robots.txt
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 200 OK

    The file is alre

In [ ]:
#import required libraries
import numpy as np
import pandas as pd
from google.colab import drive

# Base directory for MIMIC-IV dataset
BASE = '/content/drive/MyDrive/MIMIC-IV/'

#Subdirectories for different data types
ED_PATH = BASE + 'ed/'
NOTES_PATH = BASE + 'note/'
SAVE_PATH = '/content/drive/MyDrive/processed/'

In [ ]:
import os

# Check if files are still in the temp session storage
for root, dirs, files in os.walk('/content/physionet.org/'):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

# Create MIMIC-IV folder in Drive
os.makedirs('/content/drive/MyDrive/MIMIC-IV/ed/', exist_ok=True)
os.makedirs('/content/drive/MyDrive/MIMIC-IV/note/', exist_ok=True)

# Move ED files to Drive
shutil.copytree(
    '/content/physionet.org/files/mimic-iv-ed/2.2/ed/',
    '/content/drive/MyDrive/MIMIC-IV/ed/',
    dirs_exist_ok=True
)

print("Verifying")
for f in os.listdir('/content/drive/MyDrive/MIMIC-IV/ed/'):
    print(f)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Verifying
pyxis.csv.gz
medrecon.csv.gz
triage.csv.gz
vitalsign.csv.gz
edstays.csv.gz
diagnosis.csv.gz
index.html


In [ ]:
shutil.copytree(
    '/content/physionet.org/files/mimic-iv-note/2.2/note/',
    '/content/drive/MyDrive/MIMIC-IV/note/',
    dirs_exist_ok=True
)
print("Verifying...")
for f in os.listdir('/content/drive/MyDrive/MIMIC-IV/note/'):
    print(f)

Verifying...
discharge.csv.gz
discharge_detail.csv.gz
index.html
radiology.csv.gz
radiology_detail.csv.gz


In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

ED_PATH    = '/content/drive/MyDrive/MIMIC-IV/ed/'
NOTES_PATH = '/content/drive/MyDrive/MIMIC-IV/note/'

# ED tables
edstays   = pd.read_csv(ED_PATH + 'edstays.csv.gz',  parse_dates=['intime', 'outtime'])
triage    = pd.read_csv(ED_PATH + 'triage.csv.gz')
medrecon  = pd.read_csv(ED_PATH + 'medrecon.csv.gz')
vitals    = pd.read_csv(ED_PATH + 'vitalsign.csv.gz')
pyxis     = pd.read_csv(ED_PATH + 'pyxis.csv.gz')
diagnosis = pd.read_csv(ED_PATH + 'diagnosis.csv.gz')

# Notes tables
discharge_notes   = pd.read_csv(NOTES_PATH + 'discharge.csv.gz',
                                usecols=['subject_id', 'hadm_id', 'charttime', 'text'],
                                parse_dates=['charttime'])
discharge_details = pd.read_csv(NOTES_PATH + 'discharge_detail.csv.gz')


# Sanity check
for name, df in {
    'edstays': edstays, 'triage': triage, 'medrecon': medrecon,
    'vitals': vitals, 'pyxis': pyxis, 'diagnosis': diagnosis,
    'discharge_notes': discharge_notes, 'discharge_details': discharge_details
}.items():
    print(f"{name:20s} {df.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
edstays              (425087, 9)
triage               (425087, 11)
medrecon             (2987342, 9)
vitals               (1564610, 11)
pyxis                (1586053, 7)
diagnosis            (899050, 6)
discharge_notes      (331793, 4)
discharge_details    (186138, 5)


In [ ]:
# Checking target variable first
print("TARGET VARIABLE - Disposition breakdown:")
print(edstays['disposition'].value_counts())
print(f"\nMissing disposition: {edstays['disposition'].isna().sum()}")

# Checking if key linking columns exist
print("\nedstays columns:")
print(edstays.columns.tolist())

print("\ndischarge_notes columns:")
print(discharge_notes.columns.tolist())

TARGET VARIABLE - Disposition breakdown:
disposition
HOME                           241632
ADMITTED                       158010
TRANSFER                         7025
LEFT WITHOUT BEING SEEN          6155
ELOPED                           5710
OTHER                            4297
LEFT AGAINST MEDICAL ADVICE      1881
EXPIRED                           377
Name: count, dtype: int64

Missing disposition: 0

edstays columns:
['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'gender', 'race', 'arrival_transport', 'disposition']

discharge_notes columns:
['subject_id', 'hadm_id', 'charttime', 'text']


In [ ]:
disposition_map = {
    'HOME'                        : 'DISCHARGED',
    'LEFT WITHOUT BEING SEEN'     : 'DISCHARGED',
    'ELOPED'                      : 'DISCHARGED',
    'LEFT AGAINST MEDICAL ADVICE' : 'DISCHARGED',
    'ADMITTED'                    : 'ADMITTED',
    'TRANSFER'                    : 'TRANSFER',
    'OTHER'                       : 'OTHER',
    'EXPIRED'                     : 'EXPIRED'
}
print("This is just a preview - we'll apply this in the preprocessing notebook")

This is just a preview - we'll apply this in the preprocessing notebook


In [ ]:
#Merge triage (one-to-one)
df = edstays.merge(triage, on='stay_id', how='left')
print(f"After triage merge:    {df.shape}")

#Aggregate vitals per stay then merge
vitals_agg = vitals.groupby('stay_id').agg(
    temperature_mean = ('temperature', 'mean'),
    temperature_last = ('temperature', 'last'),
    heartrate_mean   = ('heartrate',   'mean'),
    heartrate_max    = ('heartrate',   'max'),
    resprate_mean    = ('resprate',    'mean'),
    o2sat_mean       = ('o2sat',       'mean'),
    o2sat_min        = ('o2sat',       'min'),
    sbp_mean         = ('sbp',         'mean'),
    dbp_mean         = ('dbp',         'mean'),
    num_vitals       = ('heartrate',   'count')
).reset_index()

df = df.merge(vitals_agg, on='stay_id', how='left')
print(f"After vitals merge:    {df.shape}")


#Aggregate diagnosis per stay then merge
diagnosis_agg = diagnosis.groupby('stay_id').agg(
    num_diagnoses = ('icd_code', 'count'),
    icd_codes     = ('icd_code', lambda x: '|'.join(x.astype(str)))
).reset_index()

df = df.merge(diagnosis_agg, on='stay_id', how='left')
print(f"After diagnosis merge: {df.shape}")


#Aggregate medrecon per stay then merge
medrecon_agg = medrecon.groupby('stay_id').agg(
    num_medications = ('name', 'count'),
).reset_index()

df = df.merge(medrecon_agg, on='stay_id', how='left')
print(f"After medrecon merge:  {df.shape}")


#Merge notes via hadm_id (keep most recent)
notes_dedup = (discharge_notes
               .sort_values('charttime')
               .groupby('hadm_id')
               .last()
               .reset_index()[['hadm_id', 'text']])

df = df.merge(notes_dedup, on='hadm_id', how='left')
print(f"After notes merge:     {df.shape}")

# Sanity checks
print(f"\nDuplicate stay_ids:  {df['stay_id'].duplicated().sum()}")
print(f"Stays with notes:    {df['text'].notna().sum()}")
print(f"Stays without notes: {df['text'].isna().sum()}")

print(f"\nMissing values (%):")
missing = df.isnull().mean().sort_values(ascending=False) * 100
print(missing[missing > 0].round(2))

After triage merge:    (425087, 19)
After vitals merge:    (425087, 29)
After diagnosis merge: (425087, 31)
After medrecon merge:  (425087, 32)
After notes merge:     (425087, 33)

Duplicate stay_ids:  0
Stays with notes:    155565
Stays without notes: 269522

Missing values (%):
text                63.40
hadm_id             52.24
num_medications     27.73
o2sat_min            6.84
o2sat_mean           6.84
temperature_last     6.13
temperature_mean     6.13
temperature          5.51
o2sat                4.85
resprate             4.79
dbp                  4.49
resprate_mean        4.45
sbp_mean             4.39
dbp_mean             4.39
heartrate_max        4.34
heartrate_mean       4.34
sbp                  4.30
heartrate            4.02
num_vitals           3.99
pain                 3.04
acuity               1.64
num_diagnoses        0.26
icd_codes            0.26
chiefcomplaint       0.01
dtype: float64


In [ ]:
SAVE_PATH = '/content/drive/MyDrive/MIMIC-IV/processed/'
import os
os.makedirs(SAVE_PATH, exist_ok=True)

# ── Stream 1: Structured data (no text column) ──────────────
df.drop(columns=['text']).to_csv(
    SAVE_PATH + 'structured_merged.csv', index=False)
print("Structured data saved ✓")

Structured data saved ✓


KeyError: "['subject_id'] not in index"

In [ ]:
SAVE_PATH = '/content/drive/MyDrive/MIMIC-IV/processed/'
#Confirm sizes
for f in os.listdir(SAVE_PATH):
    size = os.path.getsize(SAVE_PATH + f) / (1024 * 1024)
    print(f"  {f}: {size:.1f} MB")

  structured_merged.csv: 98.1 MB
  notes_merged.csv: 1715.6 MB
  eda_target_distribution.png: 0.1 MB
  eda_missing_values.png: 0.1 MB
  eda_notes_coverage.png: 0.1 MB
  eda_vitals_distribution.png: 0.1 MB
  eda_correlation_heatmap.png: 0.2 MB
  eda_acuity_analysis.png: 0.1 MB
  eda_length_of_stay.png: 0.1 MB
  eda_summary.txt: 0.0 MB
  label_mapping.json: 0.0 MB
  feature_cols.json: 0.0 MB
  structured_preprocessed.csv: 200.5 MB
  notes_preprocessed.csv: 476.6 MB


In [ ]:
# If subject_id became subject_id_x during merge
df[['stay_id', 'hadm_id', 'subject_id_x', 'disposition', 'text']].to_csv(
    SAVE_PATH + 'notes_merged.csv', index=False)
print("Notes data saved ✓")

Notes data saved ✓
